## DCR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

# --- Paramètres ---
# Longueurs d'onde (nm) pour les bandes u, g, r
wavelengths = {"u": 350, "g": 480, "r": 620, "i": 750, "z": 870, "y": 970}
# Coefficients de réfraction approximatifs (n(λ) - 1) * 1e4 pour simplifier
n_minus_1 = {"u": 3.0, "g": 2.5, "r": 2.0, "i": 1.8, "z": 1.7, "y": 1.6}  # *1e-4
# Airmass pour le template (ex: images prises à X=1.0, 1.2, 1.5)
template_airmasses = [1.0, 1.2, 1.5]
# Airmass de l'image scientifique
science_airmass = 1.1
# Position moyenne du champ (Dec = -30°)
dec = -30 * u.deg
# Échelle du détecteur (arcsec/pixel)
pixel_scale = 0.2  # arcsec/pixel


# --- Fonction pour calculer le décalage DCR ---
def dcr_shift(airmass, n_minus_1, dec, ha=0 * u.deg):
    """Calcule le décalage en RA et Dec dû à la DCR (en arcsec)."""
    # Simplification: on suppose que la DCR affecte principalement Dec (θ_y)
    # et que HA est proche de 0 (champ au méridien)
    delta_dec = n_minus_1 * airmass * np.cos(dec) * u.arcsec
    delta_ra = n_minus_1 * airmass * np.sin(dec) * np.cos(ha) * u.arcsec
    return delta_ra, delta_dec


# --- Simulation pour une étoile de couleur donnée ---
def simulate_dcr_deformation(band, template_airmasses, science_airmass, dec):
    """Simule le décalage moyen dans le template et compare à l'image scientifique."""
    # Décalage moyen dans le template (moyenne sur les airmass du template)
    delta_ra_template = (
        np.mean([dcr_shift(X, n_minus_1[band], dec)[0].value for X in template_airmasses]) * u.arcsec
    )
    delta_dec_template = (
        np.mean([dcr_shift(X, n_minus_1[band], dec)[1].value for X in template_airmasses]) * u.arcsec
    )

    # Décalage dans l'image scientifique
    delta_ra_science, delta_dec_science = dcr_shift(science_airmass, n_minus_1[band], dec)

    # Décalage résiduel (template - science)
    residual_ra = delta_ra_template - delta_ra_science
    residual_dec = delta_dec_template - delta_dec_science

    return residual_ra, residual_dec


# --- Visualisation ---
bands = ["u", "g", "r"]
residuals = {band: simulate_dcr_deformation(band, template_airmasses, science_airmass, dec) for band in bands}

# Tracer les décalages résiduels
plt.figure(figsize=(10, 5))
for band in bands:
    ra_res, dec_res = residuals[band]
    plt.scatter(
        ra_res.value,
        dec_res.value,
        label=f'Bande {band} (ΔRA={ra_res.value:.3f}", ΔDec={dec_res.value:.3f}")',
        s=100,
    )

plt.axhline(0, color="black", linestyle="--", linewidth=0.5)
plt.axvline(0, color="black", linestyle="--", linewidth=0.5)
plt.xlabel("Décalage résiduel en RA (arcsec)")
plt.ylabel("Décalage résiduel en Dec (arcsec)")
plt.title(
    "Décalage résiduel dû à la DCR (Template vs. Science)\n(Airmass template: 1.0, 1.2, 1.5 | Science: 1.1)"
)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.axis("equal")
plt.show()